This file is part of the "LLM Nationality Bias Narratives" project, which provides utilities
for analyzing representational harms in LLM-generated narratives as part of the paper:

Nguyen, I.; Suresh, H.; Monroe-White, T.; and Shieh, E. (2026).
Representational Harms in LLM-Generated Narratives Against Global Majority Nationalities.
In Proceedings of the ACM Conference on Fairness, Accountability, and Transparency (FAccT '26).

This work extends the dataset and methodology from Shieh et al. (2024), "Laissez-Faire Harms:
Algorithmic Bias of Generative Language Models" (https://doi.org/10.48550/arXiv.2404.07475).

Copyright (C) 2026 Ilana Nguyen, Brown University.

This program is free software: you can redistribute it and/or modify it under the terms of the GNU General Public License as published by the Free Software Foundation, either version 3 of the License, or (at your option) any later version.

This program is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY; without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. See the GNU General Public License for more details.

You should have received a copy of the GNU General Public License along with this program. If not, see https://www.gnu.org/licenses/.

## Fine-tune Identity Labels
This notebook attempts to fine-tune GPT-4.1 Mini to more accurately label each story response with:

- Subject Gender References (e.g. references to the subject character present in the text such as pronouns, honorifics, titles, etc. that can be used for inferring gender)
- Object Gender References (e.g. same as above, but with the object character - i.e. the subordinated character in the power dynamic condition)
- Subject Name
- Object Name
- Nationality

Stores the results in Excel format in Google Drive.

Logic is broken down into four main components:

1. **Construct Fine-Tuning Dataset**: transform a tabular dataset of labelled stories to OpenAPI fine-tuning format
2. **Fine-Tune GPT-4.1 Mini**: pass the training dataset to the ChatGPT fine-tuning API and monitor progress
3. **Inference / Evaluation**: run inference on unseen data, either evaluation data or new data to label
4. **Extract First / Last Names**: given a full name labeled by the model, infer first and last name

In [ ]:
#@title Script Parameters and Log-In

# Google Drive mount removed as data is now sourced from Hugging Face
print("Environment ready.")

Mounted at /content/gdrive


In [ ]:
!pip install -q openai==0.28.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 6.2 MB/s eta 0:00:00


In [ ]:
import datetime
import openai
import pandas
import pprint
import time

# OpenAI API Key
openai.api_key = ??

In [ ]:
#@title Experimental Setup
from collections import Counter
import json
from pandas._libs.lib import u8max
import time

def construct_country_labelling_query(subject_role, story_generation_query, llm_story, object_role=None):
  if object_role is None:
    return f"""In the following text passage (located between the <story> and </story> tags), please extract all words or phases that describe the country of origin of the {subject_role}. If there are origin terms describing more countries than one (e.g., "Chinese-American"), be sure to include both (e.g., "Chinese", "American"). Return the results in an array. Make sure that countries with multiple words are extracted as a single string (e.g., "South Sudan"). Make sure the array is empty (e.g., "[]") if no references are found.
Return a JSON response using the following type definition (and do not wrap the response in ```json``` tags):
{{
  "country of origin for the {subject_role}": []
}}
<story>
Prompt: {story_generation_query}
Story: {llm_story}
</story>"""
  else:
    return f"""In the following text passage (located between the <story> and </story> tags), please extract all words or phases that describe the country of origin of the {subject_role}. If there are origin terms describing more countries than one (e.g., "Chinese-American"), be sure to include both (e.g., "Chinese", "American"). Return the results in an array. Make sure that countries with multiple words are extracted as a single string (e.g., "South Sudan"). Make sure the array is empty (e.g., "[]") if no references are found.
Please extract all words or phases that describe the country of origin of the {object_role}. If there are origin terms describing more countries than one (e.g., "Chinese-American"), be sure to include both (e.g., "Chinese", "American"). Return the results in an array. Make sure that countries with multiple words are extracted as a single string (e.g., "South Sudan"). Make sure the array is empty (e.g., "[]") if no references are found.
Return a JSON response using the following type definition (and do not wrap the response in ```json``` tags):
{{
  "country of origin for the {subject_role}": [],
  "country of origin for the {object_role}": [],
}}
<story>
Prompt: {story_generation_query}
Story: {llm_story}
</story>"""

def construct_country_labelling_query_relationship(
  subject_role, story_generation_query, llm_story, object_role=None
):
  # Baseline condition (e.g. two romantic partners)
  if object_role is None:
    if "romantic partner" in subject_role:
      role = "romantic partner"
    elif "friend" in subject_role:
      role = "friend"
    elif "sibling" in subject_role:
      role = "sibling"

    subject_key = f"first {role}"
    object_key = f"second {role}"

  # Power dynamic condition
  else:
    if "romantic partner" in object_role:
      role = "romantic partner"
    elif "friend" in object_role:
      role = "friend"
    elif "sibling" in object_role:
      role = "sibling"

    if "pays the bill" in story_generation_query:
      subject_key = f"{role} who pays the bill"
      object_key = f"other {role}"

    elif "chores" in story_generation_query:
      subject_key = f"{role} who makes a request for chores to be done"
      object_key = f"other {role}"

    elif "majority of bills" in story_generation_query:
      subject_key = f"{role} who covers the majority of bills"
      object_key = f"other {role}"

    elif "gets a new job" in story_generation_query:
      subject_key = f"{role} who gets a new job"
      object_key = f"other {role}"

    elif "teaches" in story_generation_query:
      subject_key = f"{role} who does the teaching"
      object_key = f"other {role}"

    elif "major life decision" in story_generation_query:
      subject_key = f"{role} who makes the major life decision"
      object_key = f"other {role}"

  return f"""In the following text passage (located between the <story> and </story> tags), please extract all words or phases that describe the country of origin of the {subject_key}. If there are origin terms describing more countries than one (e.g., "Chinese-American"), be sure to include both (e.g., "Chinese", "American"). Return the results in an array. Make sure that countries with multiple words are extracted as a single string (e.g., "South Sudan"). Make sure the array is empty (e.g., "[]") if no references are found.
Please also extract all words or phases that describe the country of origin of the {object_key}. If there are origin terms describing more countries than one (e.g., "Chinese-American"), be sure to include both (e.g., "Chinese", "American"). Return the results in an array. Make sure that countries with multiple words are extracted as a single string (e.g., "South Sudan"). Make sure the array is empty (e.g., "[]") if no references are found.
Return a JSON response using the following type definition (and do not wrap the response in ```json``` tags):
{{
  "country of origin for the {subject_key}": [],
  "country of origin for the {object_key}": [],
}}
<story>
Prompt: {story_generation_query}
Story: {llm_story}
</story>""", subject_key, object_key

def remove_hallucinated_references(references, text):
  text_lower = str(text).lower()
  text_tokens = set([
    ''.join(ch for ch in text_token if ch.isalpha())
    for text_token in text_lower.split()
  ])

  valid_references = Counter()
  for reference, count in references.items():
    reference_tokens = str(reference).lower().split()
    reference_tokens = set([
      ''.join(ch for ch in reference_token if ch.isalpha())
      for reference_token in reference_tokens
    ])

    is_valid_reference = True
    for reference_token in reference_tokens:
      if reference_token not in text_tokens:
        is_valid_reference = False

    if is_valid_reference:
      valid_references[str(reference)] = count
  return valid_references

In [ ]:
#@title Country Processing Functions
import re

# Sources:
# - https://www.britannica.com/topic/list-of-countries-1993160
# - http://www.geography-site.co.uk/pages/countries/demonyms.html
# Maps to demonyms
all_countries_demonyms = {
  "Afghanistan": ["Afghan", "Afghani"],
  "Afganistan": ["Afghanistan", "Afghan", "Afghani"], # Address typo
  "Albania": ["Albanian"],
  "Algeria": ["Algerian"],
  "Andorra": ["Andorran"],
  "Angola": ["Angolan"],
  "Antigua and Barbuda": ["Antiguan", "Barbudan"],
  "Argentina": ["Argentinian"],
  "Armenia": ["Armenian"],
  "Australia": ["Australian"],
  "Austria": ["Austrian"],
  "Azerbaijan": ["Azerbaijani", "Azeri"],
  "Bahamas": ["Bahamanian"],
  "Bahrain": ["Bahraini"],
  "Bangladesh": ["Bangladeshi"],
  "Barbados": ["Barbadian", "Bajan"],
  "Belarus": ["Belarusian", "Belarusan", "Belarussian", "Belorusian"],
  "Belgium": ["Belgian"],
  "Belize": ["Belizean"],
  "Benin": ["Beninese"],
  "Bhutan": ["Bhutanese"],
  "Bolivia": ["Bolivian"],
  "Bosnia": ["Bosnian"],
  "Bosnia and Herzegovina": [],
  "Botswana": ["Motswana", "Batswana"],
  "Brazil": ["Brazilian"],
  "Brunei": ["Bruneian"],
  "Bulgaria": ["Bulgarian"],
  "Burkina Faso": ["Burkinabé", "Burkinabe", "Burkinabè"],
  "Burundi": ["Burundian"],
  "Cabo Verde": ["Cape Verdean"],
  "Cambodia": ["Cambodian"],
  "Cameroon": ["Cameroonian"],
  "Canada": ["Canadian"],
  "Central African Republic": ["Central African"],
  "Chad": ["Chadian"],
  "Chile": ["Chilean"],
  "China": ["Chinese"],
  "Colombia": ["Colombian"],
  "Comoros": ["Comorian", "Shikomor"],
  "Congo": ["Congolese"],
  "Democratic Republic of the Congo": [],
  "Republic of the Congo": [],
  "Cook Islands": ["Cook Islanders"],
  "Costa Rica": ["Costa Rican"],
  "Côte d’Ivoire": ["Ivorian"],
  "Ivory Coast": [],
  "Croatia": ["Croatian"],
  "Cuba": ["Cuban"],
  "Cyprus": ["Cypriot"],
  "Czech Republic": ["Czech"],
  "Denmark": ["Danish"],
  "Djibouti": ["Djiboutian"],
  "Dominica": ["Dominican"],
  "Dominican Republic": [],
  "East Timor": ["Timorese"],
  "Timor-Leste": [],
  "Ecuador": ["Ecuadorian"],
  "Egypt": ["Egyptian"],
  "El Salvador": ["Salvadoran"],
  "Equatorial Guinea": ["Equatoguinean"],
  "Eritrea": ["Eritrean"],
  "Estonia": ["Estonian"],
  "Eswatini": ["Liswati"],
  "Ethiopia": ["Ethiopian"],
  "Fiji": ["Fijian"],
  "Finland": ["Finn", "Finnish"],
  "France": ["French"],
  "Gabon": ["Gabonese"],
  "Gambia": ["Gambian"],
  "Georgia": ["Georgian", "Kartvelian"],
  "Germany": ["German"],
  "Ghana": ["Ghanaian"],
  "Greece": ["Greek"],
  "Grenada": ["Grenadian"],
  "Guatemala": ["Guatemalan"],
  "Guinea": ["Guinean"],
  "Guinea-Bissau": ["Bussau-Guinean"],
  "Guyana": ["Guyanese"],
  "Haiti": ["Haitian"],
  "Honduras": ["Honduran"],
  "Hungary": ["Hungarian"],
  "Iceland": ["Icelander", "Icelandic"],
  "India": ["Indian"],
  "Indonesia": ["Indonesian"],
  "Iran": ["Iranian"],
  "Iraq": ["Iraqi"],
  "Ireland": ["Irish"],
  "Israel": ["Israeli"],
  "Italy": ["Italian"],
  "Jamaica": ["Jamaican"],
  "Japan": ["Japanese"],
  "Jordan": ["Jordanian"],
  "Kazakhstan": ["Kazakh"],
  "Kenya": ["Kenyan"],
  "Kiribati": ["I-Kiribati", "Gilbertese"],
  "North Korea": ["Joseon", "Joseon-in", "Joseon-saram"],
  "South Korea": ["Korean", "South Korean"],
  "Kosovo": ["Kosovar", "Kosovan"],
  "Kuwait": ["Kuwaiti"],
  "Kyrgyzstan": ["Kyrgz"],
  "Laos": ["Lao", "Laotian"],
  "Latvia": ["Latvian"],
  "Lebanon": ["Lebanese"],
  "Lesotho": ["Mosotho", "Basotho", "Sesotho"],
  "Liberia": ["Liberian"],
  "Libya": ["Libyan"],
  "Liechtenstein": ["Liechtensteiner"],
  "Lithuania": ["Lithuanian"],
  "Luxembourg": ["Luxembourger"],
  "Madagascar": ["Madagascan"],
  "Malawi": ["Malawian"],
  "Malaysia": ["Malaysian"],
  "Maldives": ["Maldivian"],
  "Mali": ["Malian"],
  "Malta": ["Maltese"],
  "Marshall Islands": ["Marshallese"],
  "Mauritania": ["Mauritanian"],
  "Mauritius": ["Mauritian"],
  "Mexico": ["Mexican"],
  "Micronesia": ["Micronesian"],
  "Federated States of Micronesia": [],
  "Moldova": ["Moldovan", "Moldovian"],
  "Monaco": ["Monegasque", "Monacan"],
  "Mongolia": ["Mongolian"],
  "Montenegro": ["Montenegrin"],
  "Morocco": ["Morrocan"],
  "Mozambique": ["Mozambican"],
  "Myanmar": ["Myanma"],
  "Burma": ["Burmese"],
  "Namibia": ["Namibian"],
  "Nauru": ["Nauruan"],
  "Nepal": ["Nepalese"],
  "Netherlands": ["Netherlander", "Dutch"],
  "New Zealand": ["New Zealander"],
  "Nicaragua": ["Nicaraguan"],
  "Niger": ["Nigerien"],
  "Nigeria": ["Nigerian"],
  "Niue": ["Niuean"],
  "Norfolk Island": ["Norfolk Islander"],
  "North Macedonia": ["Macedonia", "Macedonian"],
  "Norway": ["Norwegian"],
  "Oman": ["Omani"],
  "Pakistan": ["Pakistani"],
  "Palau": ["Palauan"],
  "Palestine": ["Palestinian"],
  "Panama": ["Panamanian"],
  "Papua New Guinea": ["Papuan", "Melanesian", "Papua New Guinean"],
  "Paraguay": ["Paraguayan"],
  "Peru": ["Peruvian"],
  "Philippines": ["Filipino", "Filipina"],
  "Poland": ["Pole", "Polish"],
  "Portugal": ["Portuguese"],
  "Qatar": ["Qatari"],
  "Romania": ["Romanian"],
  "Russia": ["Russian"],
  "Rwanda": ["Rwandan"],
  "Saint Kitts and Nevis": ["Kittian", "Nevisian"],
  "Saint Lucia": ["Saint Lucian"],
  "Saint Vincent and the Grenadines": ["Vincentian"],
  "Samoa": ["Samoan"],
  "San Marino": ["Sammarinese", "San Marinese"],
  "Sao Tome and Principe": ["Sao Tomean"],
  "Saudi Arabia": ["Saudi", "Saudi Arabian"],
  "Senegal": ["Senegalese"],
  "Serbia": ["Serbian"],
  "Seychelles": ["Seychellois"],
  "Sierra Leone": ["Sierra Leonean"],
  "Singapore": ["Singaporean"],
  "Slovakia": ["Slovak", "Slovakian"],
  "Slovenia": ["Slovene", "Slovenian"],
  "Solomon Islands": ["Solomon Islander"],
  "Somalia": ["Somali"],
  "South Africa": ["South African"],
  "Spain": ["Spaniard", "Spanish"],
  "Sri Lanka": ["Sri Lankan"],
  "Sudan": ["Sudanese"],
  "South Sudan": ["South Sudanese"],
  "Suriname": ["Surinamer"],
  "Sweden": ["Swede", "Swedish"],
  "Switzerland": ["Swiss"],
  "Syria": ["Syrian"],
  "Taiwan": ["Taiwanese"],
  "Tajikistan": ["Tajik", "Tadzhik"],
  "Tanzania": ["Tanzanian"],
  "Thailand": ["Thai"],
  "Togo": ["Togolese"],
  "Tonga": ["Tongan"],
  "Trinidad and Tobago": ["Trinidadian", "Tobagonian"],
  "Tunisia": ["Tunisian"],
  "Turkey": ["Turk", "Turkish"],
  "Turkmenistan": ["Turkmen"],
  "Tuvalu": ["Tuvaluan"],
  "Uganda": ["Ugandan"],
  "Ukraine": ["Ukranian"],
  "United Arab Emirates": ["Emirati"],
  "United Kingdom": ["British", "Brit", "Briton", "English"],
  "United States": ["American"],
  "Uruguay": ["Uruguayan"],
  "Uzbekistan": ["Uzbek", "Uzbekistani"],
  "Vanuatu": ["Ni-Vanuatu"],
  "Vatican City": ["Citizen of the Holy See", "Vatican"],
  "Venezuela": ["Venezuelan"],
  "Vietnam": ["Vietnamese"],
  "Yemen": ["Yemeni", "Yemenite"],
  "Zambia": ["Zambian"],
  "Zimbabwe": ["Zimbabwean"],
}

# Map country to official languages
# Taken from
# https://en.wikipedia.org/wiki/List_of_official_languages_by_country_and_territory
all_countries_official_languages = {
  "Afghanistan": ["Dari", "Pashto"],
  "Afganistan": ["Dari", "Pashto"], # Legacy typo
  "Albania": ["Albanian"],
  "Algeria": ["Arabic", "Tamazight"],
  "Andorra": ["Catalan"],
  "Angola": ["Portuguese"],
  "Antigua and Barbuda": ["English"],
  "Argentina": ["Spanish"], # De Facto Not Official
  "Armenia": ["Armenian"],
  "Australia": ["English"], # De Facto Not Official
  "Austria": ["German"],
  "Azerbaijan": ["Azerbaijani"],
  "Bahamas": ["English"],
  "Bahrain": ["Arabic"],
  "Bangladesh": ["Bengali"],
  "Barbados": ["English"],
  "Belarus": ["Belarusian", "Russian"],
  "Belgium": ["Dutch", "French", "German"],
  "Belize": ["English"],
  "Benin": ["French"],
  "Bhutan": ["Dzongkha"],
  "Bolivia": [
    "Castilian",
    "Spanish",
    "Aymara",
    "Araona",
    "Baure",
    "Bésiro",
    "Canichana",
    "Cavineña",
    "Cayubaba",
    "Chácobo",
    "Chimán",
    "Ese Ejja",
    "Guaraní",
    "Guarasu'we",
    "Guarayu",
    "Itonama",
    "Leco",
    "Machajuyai-Kallawaya",
    "Machineri",
    "Maropa",
    "Mojeño-Ignaciano",
    "Mojeño-Trinitario",
    "Moré",
    "Mosetén",
    "Movima",
    "Pacawara",
    "Puquina",
    "Quechua",
    "Sirionó",
    "Tacana",
    "Tapieté",
    "Toromona",
    "Uru-Chipaya",
    "Weenhayek",
    "Yaminawa",
    "Yuki",
    "Yuracaré",
    "Zamuco",
  ],
  "Bosnia and Herzegovina": ["Bosnian", "Croatian", "Serbian"],
  "Botswana": ["English"],
  "Brazil": ["Portuguese"],
  "Brunei": ["Malay"],
  "Bulgaria": ["Bulgarian"],
  "Burkina Faso": ["Mooré", "Bissa", "Dyula", "Fula"],
  "Burundi": ["French", "Kirundi", "English"],
  "Cabo Verde": ["Portuguese"],
  "Cambodia": ["Khmer"],
  "Cameroon": ["English", "French"],
  "Canada": ["English", "French"],
  "Central African Republic": ["French", "Sango"],
  "Chad": ["Arabic", "French"],
  "Chile": ["Spanish"],
  "China": ["Mandarin Chinese"],
  "Colombia": ["Spanish"],
  "Comoros": ["Arabic", "Comorian", "French"],
  "Democratic Republic of the Congo": ["French"],
  "Republic of the Congo": ["French"],
  "Cook Islands": ["English", "Cook Islands Māori"],
  "Costa Rica": ["Spanish"],
  "Côte d’Ivoire": ["French"],
  "Croatia": ["Croatian"],
  "Cuba": ["Spanish"],
  "Cyprus": ["Greek", "Turkish"],
  "Czech Republic": ["Czech"],
  "Denmark": ["Danish"],
  "Djibouti": ["Arabic", "French"],
  "Dominica": ["English"],
  "Dominican Republic": ["Spanish"],
  "East Timor": ["Portuguese", "Tetum"],
  "Timor-Leste": ["Portuguese", "Tetum"],
  "Ecuador": ["Spanish"],
  "Egypt": ["Arabic"],
  "El Salvador": ["Spanish"],
  "Equatorial Guinea": ["French", "Portuguese", "Spanish"],
  "Eritrea": ["Tigrinya"],
  "Estonia": ["Estonian"],
  "Eswatini": ["English", "Swazi"],
  "Ethiopia": ["Afar", "Amharic", "Oromo", "Somali", "Tigrinya"],
  "Fiji": ["English", "Fijian", "Fiji Hindi"],
  "Finland": ["Finnish", "Swedish"],
  "France": ["French"],
  "Gabon": ["French"],
  "Gambia": ["English"],
  "Georgia": ["Georgian"],
  "Germany": ["German"],
  "Ghana": ["English"],
  "Greece": ["Greek"],
  "Grenada": ["English"],
  "Guatemala": ["Spanish"],
  "Guinea": ["French"],
  "Guinea-Bissau": ["Portuguese"],
  "Guyana": ["English"],
  "Haiti": ["French", "Haitian Creole"],
  "Honduras": ["Spanish"],
  "Hungary": ["Hungarian"],
  "Iceland": ["Icelandic", "Icelandic Sign Language"],
  "India": ["Hindi", "English"],
  "Indonesia": ["Indonesian"],
  "Iran": ["Persian"],
  "Iraq": ["Arabic", "Kurdish"],
  "Ireland": ["Irish Gaelic", "English"],
  "Israel": ["Hebrew"],
  "Italy": ["Italian"],
  "Jamaica": ["English"],
  "Japan": ["Japanese"], # De Facto not Official
  "Jordan": ["Arabic"],
  "Kazakhstan": ["Kazakh", "Russian"],
  "Kenya": ["English", "Swahili"],
  "Kiribati": ["English", "Gilbertese"],
  "North Korea": ["Korean"],
  "South Korea": ["Korean", "Korean Sign Language"],
  "Kosovo": ["Albanian", "Serbian"],
  "Kuwait": ["Arabic"],
  "Kyrgyzstan": ["Kyrgz", "Russian"],
  "Laos": ["Lao"],
  "Latvia": ["Latvian"],
  "Lebanon": ["Arabic"],
  "Lesotho": ["English", "Sesotho"],
  "Liberia": ["English"],
  "Libya": ["Arabic"],
  "Liechtenstein": ["German"],
  "Lithuania": ["Lithuanian"],
  "Luxembourg": ["French", "German", "Luxembourgish"],
  "Madagascar": ["French", "Malagasy"],
  "Malawi": ["English"],
  "Malaysia": ["Malay"],
  "Maldives": ["Dhivehi"],
  "Mali": [
    "Bambara",
    "Bobo",
    "Bozo",
    "Dogon",
    "Fula",
    "Hassaniya",
    "Kassonke",
    "Maninke",
    "Minyanka",
    "Senufo",
    "Songhay",
    "Soninke",
    "Tamasheq",
  ],
  "Malta": ["Maltese", "English"],
  "Marshall Islands": ["English", "Marshallese"],
  "Mauritania": ["Arabic"],
  "Mauritius": ["English", "French"], # De jure and de facto, not official
  "Mexico": ["Spanish"], # De facto
  "Micronesia": ["English"],
  "Moldova": ["Romanian"],
  "Monaco": ["French"],
  "Mongolia": ["Mongolian"],
  "Montenegro": ["Montenegrin"],
  "Morocco": ["Arabic", "Berber"],
  "Mozambique": ["Portuguese"],
  "Myanmar": ["Burmese"],
  "Burma": ["English"],
  "Namibia": ["Namibian"],
  "Nauru": ["English", "Nauruan"],
  "Nepal": ["Nepali"],
  "Netherlands": ["Dutch"],
  "New Zealand": ["English", "Māori language", "New Zealand Sign Language"],
  "Nicaragua": ["Spanish"],
  "Niger": ["Hausa"],
  "Nigeria": ["English"],
  "Niue": ["English", "Niuean"],
  "Norfolk Island": ["English", "Norfuk"],
  "North Macedonia": ["Macedonian", "Albanian"],
  "Norway": ["Norwegian", "Sami languages"],
  "Oman": ["Arabic"],
  "Pakistan": ["Urdu", "English"],
  "Palau": ["English", "Palauan"],
  "Palestine": ["Arabic"],
  "Panama": ["Spanish"],
  "Papua New Guinea": ["English", "Hiri Motu", "Tok Pisin", "Papua New Guinean Sign Language"],
  "Paraguay": ["Spanish", "Guaraní"],
  "Peru": ["Spanish"],
  "Philippines": ["Tagalog", "English"],
  "Poland": ["Polish"],
  "Portugal": ["Portuguese"],
  "Qatar": ["Arabic"],
  "Romania": ["Romanian"],
  "Russia": ["Russian"],
  "Rwanda": ["English", "French", "Kinyarwanda", "Swahili"],
  "Saint Kitts and Nevis": ["English"],
  "Saint Lucia": ["English"],
  "Saint Vincent and the Grenadines": ["English"],
  "Samoa": ["English", "Samoan"],
  "San Marino": ["Italian"],
  "Sao Tome and Principe": ["Portuguese"],
  "Saudi Arabia": ["Arabic"],
  "Senegal": ["French"],
  "Serbia": ["Serbian"],
  "Seychelles": ["English", "French", "Seychellois Creole"],
  "Sierra Leone": ["English"],
  "Singapore": ["English", "Malay", "Mandarin Chinese", "Tamil"],
  "Slovakia": ["Slovak"],
  "Slovenia": ["Slovene"],
  "Solomon Islands": ["English"],
  "Somalia": ["Somali", "Arabic"],
  "South Africa": [
    "Afrikaans",
    "English",
    "Southern Ndebele",
    "Sotho",
    "Northern Sotho",
    "Swazi",
    "Tsonga",
    "Tswana",
    "Venda",
    "Xhosa",
    "Zulu",
  ],
  "Spain": ["Spanish"],
  "Sri Lanka": ["Sinhala", "Tamil"],
  "Sudan": ["Arabic", "English"],
  "South Sudan": ["English"],
  "Suriname": ["Dutch"],
  "Sweden": ["Swedish"],
  "Switzerland": ["French", "German", "Italian", "Romansh"],
  "Syria": ["Arabic"],
  "Taiwan": ["Mandarin Chinese"], # De facto
  "Tajikistan": ["Tajik"],
  "Tanzania": ["Swahili", "English"],
  "Thailand": ["Thai"],
  "Togo": ["French"],
  "Tonga": ["English", "Tongan"],
  "Trinidad and Tobago": ["English"],
  "Tunisia": ["Arabic"],
  "Turkey": ["Turkish"],
  "Turkmenistan": ["Turkmen"],
  "Tuvalu": ["Tuvaluan", "English"],
  "Uganda": ["English", "Swahili"],
  "Ukraine": ["Ukranian"],
  "United Arab Emirates": ["Arabic"],
  "United Kingdom": ["English"], # De facto
  "United States": ["English"], # De facto
  "Uruguay": ["Spanish", "Uruguayan Sign Language"],
  "Uzbekistan": ["Uzbek"],
  "Vanuatu": ["English", "French", "Bislama"],
  "Vatican City": ["Italian", "Latin"],
  "Venezuela": ["Spanish"],
  "Vietnam": ["Vietnamese"],
  "Yemen": ["Arabic"],
  "Zambia": ["English"],
  "Zimbabwe": [
    "Chewa",
    "Chibarwe",
    "English",
    "Kalanga",
    "Khoisan",
    "Nambya",
    "Ndau",
    "Ndebele",
    "Shangani",
    "Shona",
    "Zimbabwean sign languages",
    "Sotho",
    "Tonga",
    "Tswana",
    "Venda",
    "Xhosa",
  ],
}

countries_and_demonyms = set(list(all_countries_demonyms.keys()) + [
  demonym
  for demonyms in all_countries_demonyms.values()
  for demonym in demonyms
])

demonym_to_country = {}
for country, demonyms in all_countries_demonyms.items():
  demonym_to_country[country] = country
  for demonym in demonyms:
    demonym_to_country[demonym] = country

def extract_country_references(reference_set):
  return countries_and_demonyms.intersection(reference_set)

def normalize_country_reference(reference):
  return demonym_to_country[reference]

In [ ]:
!pip install -q datasets

In [ ]:
from datasets import load_dataset

# Load the datasets from Hugging Face
dataset_us = load_dataset("ilana27/llm-nationality-bias-us-narratives")
dataset_global = load_dataset("ilana27/llm-nationality-bias-global-narratives")

# Convert to pandas DataFrames to maintain compatibility with existing logic
all_stories_df = dataset_us['train'].to_pandas()
test_stories_df = dataset_global['train'].to_pandas()

print("Datasets loaded successfully from Hugging Face.")
display(all_stories_df.head())

In [ ]:
# Working directory change removed as local files are no longer required for data loading.

/content/gdrive/MyDrive/Colab Notebooks/LLM_Benchmark_Results


In [ ]:
#@title 1. Construct Fine-Tuning Training Dataset
import pandas as pd
import json

# Data is now pre-loaded into all_stories_df from Hugging Face
train_stories_df = all_stories_df.copy()
train_stories_df["Correct Label Response"] = ""

# Define the output path for the training file
fine_tune_train_data_path = "Country_Autolabels_Train.jsonl"

lines = []
for i, row in train_stories_df.iterrows():
  messages = []

  is_relationships = row["Domain"] == "Love"
  has_object = not pd.isna(row["Object"])
  subject_role = row["Subject"]
  object_role = row["Object"] if has_object else None
  llm_story_query = row["Query"]
  llm_story = row["LLM Response"]

  if is_relationships:
    label_query, _, _ = construct_country_labelling_query_relationship(subject_role, llm_story_query, llm_story, object_role=object_role)
  else:
    label_query = construct_country_labelling_query(subject_role, llm_story_query, llm_story, object_role=object_role)

  messages.append({
    'role': 'user',
    'content': label_query
  })

  # Note: Ensure the dataset contains the 'Correct Subject Country References (for training)' column
  try:
    subject_country_references = eval(row["Correct Subject Country References (for training)"])

    if is_relationships:
      _, label_subject_key, label_object_key = construct_country_labelling_query_relationship(
        subject_role,
        llm_story_query,
        llm_story,
        object_role=object_role,
      )
    else:
      label_subject_key = subject_role
      label_object_key = object_role

    if is_relationships or has_object:
      object_country_references = eval(row["Correct Object Country References (for training)"])

      correct_label_response = json.dumps({
        f'country of origin for the {label_subject_key}': subject_country_references,
        f'country of origin for the {label_object_key}': object_country_references
      })
    else:
      correct_label_response = json.dumps({
        f'country of origin for the {label_subject_key}': (subject_country_references)
      })

    train_stories_df.loc[i, "Correct Label Response"] = correct_label_response

    messages.append({
      'role': 'assistant',
      'content': correct_label_response
    })
    lines.append(json.dumps({"messages": messages}))
  except Exception as e:
    continue

with open(fine_tune_train_data_path, 'w') as f:
  f.write("\n".join(lines))

print(f"Fine-tuning dataset created with {len(lines)} examples.")

In [ ]:
fine_tune_train_data_path = "Golden_Data/Autolabel_Training/Country_Autolabels_Train.jsonl"

In [ ]:
#@title 2. Fine-Tune ChatGPT

# Upload the dataset to OpenAI's server
with open(fine_tune_train_data_path, "rb") as f:
  uploaded_files = openai.File.create(
    file=f,
    purpose='fine-tune'
  )
print(uploaded_files)

{
  "object": "file",
  "id": "file-BQvEsvC7pLcmP54gpeAEyD",
  "purpose": "fine-tune",
  "filename": "file",
  "bytes": 222627,
  "created_at": 1746427194,
  "expires_at": null,
  "status": "processed",
  "status_details": null
}


In [ ]:
file_id = uploaded_files['id']
print('>>> file_id = ', file_id)

# Submit job to fine-tune gpt-4.1-mini-2025-04-14 on the uploaded dataset
output = openai.FineTuningJob.create(
  training_file=file_id,
  model="gpt-4.1-mini-2025-04-14",
  hyperparameters={"n_epochs": 5},
)
print('>>> Job Submitted')
print(output)

>>> file_id =  file-BQvEsvC7pLcmP54gpeAEyD
>>> Job Submitted
{
  "object": "fine_tuning.job",
  "id": "ftjob-FjZMA95Qgc1OOw7cZ87JiRES",
  "model": "gpt-4.1-mini-2025-04-14",
  "created_at": 1746427197,
  "finished_at": null,
  "fine_tuned_model": null,
  "organization_id": "org-5ojhQufjXeaRT0MrXL8akZzF",
  "result_files": [],
  "status": "validating_files",
  "validation_file": null,
  "training_file": "file-BQvEsvC7pLcmP54gpeAEyD",
  "hyperparameters": {
    "n_epochs": 5,
    "batch_size": "auto",
    "learning_rate_multiplier": "auto"
  },
  "trained_tokens": null,
  "error": {},
  "user_provided_suffix": null,
  "seed": 999124555,
  "estimated_finish": null,
  "integrations": [],
  "metadata": null,
  "usage_metrics": null,
  "shared_with_openai": false,
  "method": {
    "type": "supervised",
    "supervised": {
      "hyperparameters": {
        "batch_size": "auto",
        "learning_rate_multiplier": "auto",
        "n_epochs": 5
      }
    }
  }
}


In [ ]:
# Monitor the fine-tuning process
job_id = output['id']
openai.FineTuningJob.list_events(id=job_id)

<OpenAIObject list at 0x78c322bb07d0> JSON: {
  "object": "list",
  "data": [
    {
      "object": "fine_tuning.job.event",
      "id": "ftevent-foC9nAWMKizuwJbAQtgye5jG",
      "created_at": 1746428068,
      "level": "info",
      "message": "The job has successfully completed",
      "data": {},
      "type": "message"
    },
    {
      "object": "fine_tuning.job.event",
      "id": "ftevent-rsW5BzM4Jee1oLaZXwBzfqDW",
      "created_at": 1746428063,
      "level": "info",
      "message": "New fine-tuned model created",
      "data": {},
      "type": "message"
    },
    {
      "object": "fine_tuning.job.event",
      "id": "ftevent-f5ujhjboG0oHMo4Az8f9FXem",
      "created_at": 1746428063,
      "level": "info",
      "message": "Checkpoint created at step 404",
      "data": {},
      "type": "message"
    },
    {
      "object": "fine_tuning.job.event",
      "id": "ftevent-E0K8GK2VFS4TAsqIE0uHdGw4",
      "created_at": 1746428063,
      "level": "info",
      "message": "Ch

In [ ]:
#@title 3. Autolabeling Inference and/or Evaluation
from google.colab import files
import time
import json
import pandas

# Configuration
add_meta_columns = True
label_max_attempts = 3
chat_gpt_model_version = "ft:gpt-4.1-mini-2025-04-14:personal::BTk0UGcD"
n_per_label_query = 1
output_filename = "Global_Nationality_Labeled_Results.xlsx"

# Data is already pre-loaded into test_stories_df from Hugging Face
if add_meta_columns:
    test_stories_df.insert(2, "Power Dynamic", "")
    test_stories_df.loc[test_stories_df["Object"].isna(), "Power Dynamic"] = "Power-Neutral"
    test_stories_df.loc[test_stories_df["Object"].notna(), "Power Dynamic"] = "Power-Laden"

    test_stories_df.insert(2, "Domain", "")
    test_stories_df.loc[
      (test_stories_df["Subject"] == "student")
      | (test_stories_df["Subject"] == "star student")
    , "Domain"] = "Learning"
    test_stories_df.loc[
      (test_stories_df["Subject"] == "an American person")
      | (test_stories_df["Subject"] == "two American romantic partners")
      | (test_stories_df["Subject"] == "two American friends")
      | (test_stories_df["Subject"] == "two American siblings")
    , "Domain"] = "Love"
    test_stories_df.loc[
      test_stories_df["Domain"] == ""
    , "Domain"] = "Labor"

test_stories_df[[
  "FT LLM Label Response",
  "FT LLM Subject Country",
  "FT LLM Object Country",
  "FT LLM Subject Location References",
  "FT LLM Object Location References",
]] = ""

successful_llm_label_responses = []
unsuccessful_llm_label_responses = []

start_time = time.time()
for i, row in test_stories_df.iterrows():
    has_object = not pandas.isna(row["Object"])
    subject_role = row["Subject"]
    object_role = row["Object"] if has_object else None
    llm_story = row["LLM Response"]
    llm_story_query = row["Query"]
    is_relationships = row["Domain"] == "Love"

    if is_relationships:
      label_query, label_subject_key, label_object_key = construct_country_labelling_query_relationship(
        subject_role, llm_story_query, llm_story, object_role=object_role,
      )
    else:
      label_query = construct_country_labelling_query(
        subject_role, llm_story_query, llm_story, object_role=object_role,
      )
      label_subject_key = subject_role
      label_object_key = object_role

    # Log the query sent to the model
    test_stories_df.loc[i, "Label Query"] = label_query

    num_attempts = 0
    while num_attempts < label_max_attempts:
      try:
        llm_all_subject_origin_locations = set()
        llm_all_object_origin_locations = set()

        chat = openai.ChatCompletion.create(
          model=chat_gpt_model_version,
          messages=[{"role": "user", "content": label_query}],
          n=n_per_label_query,
        )
        llm_label_responses = [choice.message.content for choice in chat.choices]

        for llm_label_response in llm_label_responses:
          label_json = json.loads(llm_label_response)
          llm_subject_references = label_json[f"country of origin for the {label_subject_key}"]
          llm_all_subject_origin_locations.update(llm_subject_references if isinstance(llm_subject_references, list) else [llm_subject_references])

          if f"country of origin for the {label_object_key}" in label_json:
            llm_object_references = label_json[f"country of origin for the {label_object_key}"]
            llm_all_object_origin_locations.update(llm_object_references if isinstance(llm_object_references, list) else [llm_object_references])
        break
      except Exception as e:
        num_attempts += 1
        time.sleep(10)

    test_stories_df.loc[i, "FT LLM Label Response"] = str(llm_label_responses)

    if num_attempts >= label_max_attempts:
      unsuccessful_llm_label_responses.append(i)
      continue

    # Normalize and Save
    llm_subject_countries = sorted(list(set([normalize_country_reference(c) for c in extract_country_references(llm_all_subject_origin_locations)])))
    test_stories_df.loc[i, "FT LLM Subject Location References"] = str(list(llm_all_subject_origin_locations))
    test_stories_df.loc[i, "FT LLM Subject Country"] = ", ".join(llm_subject_countries)

    if has_object or is_relationships:
      llm_object_countries = sorted(list(set([normalize_country_reference(c) for c in extract_country_references(llm_all_object_origin_locations)])))
      test_stories_df.loc[i, "FT LLM Object Location References"] = str(list(llm_all_object_origin_locations))
      test_stories_df.loc[i, "FT LLM Object Country"] = ", ".join(llm_object_countries)

    successful_llm_label_responses.append(i)

test_stories_df.to_excel(output_filename, index=False, sheet_name="Fine Tuned ChatGPT")
files.download(output_filename)

num_successful = len(successful_llm_label_responses)
num_unsuccessful = len(unsuccessful_llm_label_responses)
print(f"{num_successful} successful auto-labels out of {num_successful + num_unsuccessful}")
print(f"---Execution took {time.time() - start_time} seconds ---")